In [44]:
from import_images import encontrar_imagens_tiff, carregar_imagem_por_indice
from cellpose import models, utils
from skimage.exposure import equalize_adapthist
from skimage.morphology import white_tophat, closing, disk, label
from skimage.measure import regionprops
from scipy.ndimage import gaussian_filter
import numpy as np
import matplotlib.pyplot as plt
import os


class ProcessadorDeImagens:
    def __init__(self, base_dir, modelo='nuclei', canais=[0, 0]):
        self.image_paths = encontrar_imagens_tiff(base_dir)
        self.modelo = models.Cellpose(model_type=modelo)
        self.canais = canais
        self.output_dir = os.path.join(base_dir, "segmentacoes")
        os.makedirs(self.output_dir, exist_ok=True)

    def preprocessar_imagem(self, imagem):
        imagem = gaussian_filter(imagem, sigma=1)
        imagem = white_tophat(imagem, footprint=disk(15))
        imagem = equalize_adapthist(imagem, clip_limit=0.4)
        imagem = (imagem - np.min(imagem)) / (np.max(imagem) - np.min(imagem))
        imagem = closing(imagem, disk(3))
        return imagem

    def filtrar_objetos(self, mascara, imagem_original, limiar_circularidade=0.75, limiar_area=80, fator_intensidade=1.2):
        nova_mascara = np.zeros_like(mascara)
        props = regionprops(label(mascara), intensity_image=imagem_original)

        media_imagem = np.mean(imagem_original)
        index = 1
        for prop in props:
            if prop.perimeter == 0:
                continue

            circularidade = (4 * np.pi * prop.area) / (prop.perimeter ** 2)
            intensidade_media = prop.mean_intensity
            area = prop.area

            if (
                circularidade >= limiar_circularidade and
                area >= limiar_area and
                intensidade_media > media_imagem * fator_intensidade
            ):
                nova_mascara[label(mascara) == prop.label] = index
                index += 1

        return nova_mascara

    def carregar_e_processar(self, indice):
        imagem_original = carregar_imagem_por_indice(self.image_paths, indice)
        if imagem_original is None:
            print("Não foi possível carregar a imagem.")
            return None, None

        print("Pré-processando imagem...")
        imagem = self.preprocessar_imagem(imagem_original)

        print("Segmentando com Cellpose...")
        masks, flows, styles, diams = self.modelo.eval(
            imagem,
            channels=self.canais,
            flow_threshold=1,
            cellprob_threshold=-0.1,
            min_size=300
        )

        print("Filtrando objetos...")
        masks_filtradas = self.filtrar_objetos(
            mascara=masks,
            imagem_original=imagem_original,
            limiar_circularidade=0.75,
            limiar_area=80,
            fator_intensidade=1.2
        )

        n_objetos = len(np.unique(masks_filtradas)) - 1
        print(f"{n_objetos} objetos circulares mantidos")

        outlines = utils.outlines_list(masks_filtradas)
        nome_arquivo = os.path.basename(self.image_paths[indice])
        nome_base = os.path.splitext(nome_arquivo)[0]
        caminho_saida = os.path.join(self.output_dir, f"{nome_base}_segmentado.png")

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(imagem_original, cmap='gray')
        for o in outlines:
            ax.plot(o[:, 0], o[:, 1], color='red', linewidth=0.5)

        ax.set_title(f"{nome_base} - {n_objetos} objetos circulares")
        ax.axis('off')
        plt.savefig(caminho_saida, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Segmentação salva em: {caminho_saida}")
        return masks_filtradas, flows


if __name__ == "__main__":
    base_dir = os.getcwd()
    proc = ProcessadorDeImagens(base_dir, modelo='nuclei')
    masks, flows = proc.carregar_e_processar(0)


Imagem carregada: /home/kayllany.oliveira/IC/19 Resultados HTS MAYV/MAYV HTS TargetMol_R2/M_20_R2[10510]/2022-06-07T164120Z[11174]/008012-1-001001001.tif - Dimensão: (1024, 1360)
Pré-processando imagem...
Segmentando com Cellpose...
Filtrando objetos...
12 objetos circulares mantidos
Segmentação salva em: /home/kayllany.oliveira/IC/segmentacoes/008012-1-001001001_segmentado.png


In [ ]:
from import_images import encontrar_imagens_tiff, carregar_imagem_por_indice
from skimage.exposure import equalize_adapthist
from skimage.morphology import white_tophat, closing, disk, label
from skimage.measure import regionprops
from scipy.ndimage import gaussian_filter
from csbdeep.utils import normalize
from stardist.models import StarDist2D
from cellpose import models as cp_models, utils as cp_utils
import numpy as np
import matplotlib.pyplot as plt
import os


class ProcessadorDeImagens:
    def __init__(self, base_dir, metodo_segmentacao='cellpose', canais=[0, 0]):
        self.image_paths = encontrar_imagens_tiff(base_dir)
        self.metodo = metodo_segmentacao.lower()
        self.canais = canais
        self.output_dir = os.path.join(base_dir, "segmentacoes")
        os.makedirs(self.output_dir, exist_ok=True)

        if self.metodo == 'cellpose':
            self.modelo = cp_models.Cellpose(model_type='nuclei')
        elif self.metodo == 'stardist':
            self.modelo = StarDist2D.from_pretrained('2D_versatile_fluo')
        else:
            raise ValueError("Método de segmentação não reconhecido. Use 'cellpose' ou 'stardist'.")

    def preprocessar_imagem(self, imagem):
        imagem = gaussian_filter(imagem, sigma=1)
        imagem = white_tophat(imagem, footprint=disk(15))
        imagem = equalize_adapthist(imagem, clip_limit=0.4)
        imagem = (imagem - np.min(imagem)) / (np.max(imagem) - np.min(imagem))
        imagem = closing(imagem, disk(3))
        return imagem

    def segmentar_imagem(self, imagem):
        if self.metodo == 'cellpose':
            masks, flows, styles, diams = self.modelo.eval(
                imagem,
                channels=self.canais,
                flow_threshold=1,
                cellprob_threshold=-0.1,
                min_size=150
            )
            return masks
        elif self.metodo == 'stardist':
            imagem_norm = normalize(imagem, 1, 99.8, clip=True)
            labels, _ = self.modelo.predict_instances(imagem_norm)
            return labels

    def filtrar_objetos(self, mascara, imagem_original, limiar_circularidade=0.75, limiar_area=80, fator_intensidade=1.2):
        nova_mascara = np.zeros_like(mascara)
        props = regionprops(label(mascara), intensity_image=imagem_original)

        media_imagem = np.mean(imagem_original)
        index = 1
        for prop in props:
            if prop.perimeter == 0:
                continue
            circularidade = (4 * np.pi * prop.area) / (prop.perimeter ** 2)
            if (
                circularidade >= limiar_circularidade and
                prop.area >= limiar_area and
                prop.mean_intensity > media_imagem * fator_intensidade
            ):
                nova_mascara[label(mascara) == prop.label] = index
                index += 1

        return nova_mascara

    def carregar_e_processar(self, indice):
        imagem_original = carregar_imagem_por_indice(self.image_paths, indice)
        if imagem_original is None:
            print("Não foi possível carregar a imagem.")
            return None, None

        print("Pré-processando imagem...")
        imagem = self.preprocessar_imagem(imagem_original)

        print(f"Segmentando com {self.metodo.title()}...")
        masks = self.segmentar_imagem(imagem)

        print("Filtrando objetos...")
        masks_filtradas = self.filtrar_objetos(masks, imagem_original)

        n_objetos = len(np.unique(masks_filtradas)) - 1
        print(f"{n_objetos} objetos circulares mantidos")

        nome_arquivo = os.path.basename(self.image_paths[indice])
        nome_base = os.path.splitext(nome_arquivo)[0]
        caminho_saida = os.path.join(self.output_dir, f"{nome_base}_segmentado.png")

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(imagem_original, cmap='gray')

        outlines = cp_utils.outlines_list(masks_filtradas)
        for o in outlines:
            ax.plot(o[:, 0], o[:, 1], color='red', linewidth=0.5)

        ax.set_title(f"{nome_base} - {n_objetos} objetos circulares")
        ax.axis('off')
        plt.savefig(caminho_saida, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Segmentação salva em: {caminho_saida}")
        return masks_filtradas


if __name__ == "__main__":
    base_dir = os.getcwd()
    proc = ProcessadorDeImagens(base_dir, metodo_segmentacao='stardist')  # ou 'cellpose'
    masks = proc.carregar_e_processar(250)
